# CNN–VGG16 — Modelo B (Test)

Variación directa de `VGG16_Test.ipynb`: evaluación completa y predicción individual.


In [ ]:
# ADAPTADO: evaluación completa multiclase
!pip -q install tensorflow-datasets==4.9.6 scikit-learn==1.5.2 seaborn==0.13.2
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.model_selection import train_test_split

SEMILLA = 42
TAMANIO_IMAGEN = (224, 224)
BATCH_SIZE = 16
RAIZ = Path.cwd()
for carpeta in ["data", "results", "figures", "weights"]:
    (RAIZ / carpeta).mkdir(exist_ok=True)


In [ ]:
# Cargar exactamente el mismo conjunto y reconstruir la partición de prueba
dataset_base, info = tfds.load(
    "tf_flowers:3.0.1", split="train", as_supervised=True,
    with_info=True, shuffle_files=False
)
CLASES = list(info.features["label"].names)
TOTAL = info.splits["train"].num_examples
etiquetas = np.fromiter((int(y) for _, y in tfds.as_numpy(dataset_base)), dtype=np.int64, count=TOTAL)
indices = np.arange(TOTAL, dtype=np.int64)
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, etiquetas, test_size=0.30, random_state=SEMILLA, stratify=etiquetas
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, random_state=SEMILLA, stratify=y_temp
)
def crear_split(indices_split):
    llaves = tf.constant(indices_split, dtype=tf.int64)
    tabla = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(llaves, tf.ones_like(llaves, dtype=tf.int32)), 0
    )
    sin_batch = dataset_base.enumerate().filter(
        lambda i, elemento: tabla.lookup(i) > 0
    ).map(lambda i, elemento: elemento)
    preparado = sin_batch.map(
        lambda x, y: (tf.image.resize(tf.image.convert_image_dtype(x, tf.float32), TAMANIO_IMAGEN), y),
        num_parallel_calls=tf.data.AUTOTUNE
    ).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return sin_batch, preparado

validacion_sin_batch, validacion = crear_split(idx_val)
prueba_sin_batch, prueba = crear_split(idx_test)


In [ ]:
# Cargar el modelo guardado por el Colab Train correspondiente
ruta_modelo = RAIZ / "weights" / "modelo_B.keras"
if not ruta_modelo.exists():
    raise FileNotFoundError("Ejecuta primero VGG16_Modelo_B_Train.ipynb y coloca sus artefactos en weights/.")
modelo = tf.keras.models.load_model(ruta_modelo)

# Predicciones sobre TODO el conjunto de prueba
y_real = np.concatenate([y.numpy() for _, y in prueba])
probabilidades = modelo.predict(prueba, verbose=1)
y_pred = probabilidades.argmax(axis=1)
confianza = probabilidades.max(axis=1)
assert len(y_real) == len(idx_test) and probabilidades.shape == (len(y_real), 5)
y_val_real = np.concatenate([y.numpy() for _, y in validacion])
y_val_pred = modelo.predict(validacion, verbose=0).argmax(axis=1)
_, _, f1_validacion, _ = precision_recall_fscore_support(
    y_val_real, y_val_pred, average="macro", zero_division=0
)

predicciones = pd.DataFrame({
    "index": np.sort(idx_test), "real": y_real, "predicha": y_pred,
    "clase_real": [CLASES[i] for i in y_real],
    "clase_predicha": [CLASES[i] for i in y_pred],
    "confianza": confianza,
})
for i, clase in enumerate(CLASES):
    predicciones[f"p_{clase}"] = probabilidades[:, i]
predicciones.to_csv(RAIZ / "results" / "modelo_B_predicciones.csv", index=False)

cm = confusion_matrix(y_real, y_pred, labels=range(5))
cm_norm = confusion_matrix(y_real, y_pred, labels=range(5), normalize="true")
precision, recall, f1, soporte = precision_recall_fscore_support(
    y_real, y_pred, labels=range(5), zero_division=0
)
tn = cm.sum() - (cm.sum(axis=0) + cm.sum(axis=1) - np.diag(cm))
fp = cm.sum(axis=0) - np.diag(cm)
fn = cm.sum(axis=1) - np.diag(cm)
tp = np.diag(cm)
por_clase = pd.DataFrame({
    "clase": CLASES, "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    "precision": precision, "recall": recall, "f1": f1, "soporte": soporte,
})
por_clase.to_csv(RAIZ / "results" / "modelo_B_metricas_por_clase.csv", index=False)
pd.DataFrame(cm, index=CLASES, columns=CLASES).to_csv(RAIZ / "results" / "modelo_B_matriz_confusion.csv")
pd.DataFrame(cm_norm, index=CLASES, columns=CLASES).to_csv(RAIZ / "results" / "modelo_B_matriz_confusion_normalizada.csv")

p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y_real, y_pred, average="macro", zero_division=0)
p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(y_real, y_pred, average="weighted", zero_division=0)
metricas = {
    "modelo": "B", "accuracy": accuracy_score(y_real, y_pred),
    "precision_macro": p_macro, "recall_macro": r_macro, "f1_macro": f_macro,
    "precision_weighted": p_weighted, "recall_weighted": r_weighted, "f1_weighted": f_weighted,
    "f1_macro_validacion": f1_validacion,
    "brecha_f1_validacion_prueba": abs(f1_validacion - f_macro),
}
assert np.isclose(metricas["accuracy"], np.trace(cm) / cm.sum())
for i in range(5):
    assert np.isclose(precision[i], tp[i] / (tp[i] + fp[i]) if tp[i] + fp[i] else 0)
    assert np.isclose(recall[i], tp[i] / (tp[i] + fn[i]) if tp[i] + fn[i] else 0)
(RAIZ / "results" / "modelo_B_metricas.json").write_text(json.dumps(metricas, indent=2))
(RAIZ / "results" / "modelo_B_classification_report.txt").write_text(
    classification_report(y_real, y_pred, target_names=CLASES, zero_division=0)
)
print(json.dumps(metricas, indent=2))
display(por_clase)


In [ ]:
# Matriz de confusión absoluta y normalizada
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES, ax=axes[0])
axes[0].set(title="Matriz de confusión — Modelo B", xlabel="Predicción", ylabel="Observación")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", xticklabels=CLASES, yticklabels=CLASES, ax=axes[1])
axes[1].set(title="Matriz normalizada — Modelo B", xlabel="Predicción", ylabel="Observación")
for ax in axes: ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(RAIZ / "figures" / "matrices_confusion_modelo_B.png", dpi=190)
plt.show()

# Precision, Recall y F1 por clase
por_clase.set_index("clase")[["precision", "recall", "f1"]].plot(kind="bar", figsize=(11, 5), ylim=(0, 1))
plt.title("Métricas por clase — Modelo B")
plt.ylabel("Valor"); plt.xticks(rotation=20); plt.grid(axis="y", alpha=.25); plt.tight_layout()
plt.savefig(RAIZ / "figures" / "metricas_clase_modelo_B.png", dpi=180)
plt.show()

# Histograma de confianza de aciertos y errores
predicciones["resultado"] = np.where(y_real == y_pred, "Correcta", "Incorrecta")
plt.figure(figsize=(9, 5))
sns.histplot(data=predicciones, x="confianza", hue="resultado", bins=20, multiple="layer")
plt.title("Confianza de predicciones — Modelo B")
plt.xlim(0, 1); plt.tight_layout()
plt.savefig(RAIZ / "figures" / "histograma_confianza_modelo_B.png", dpi=180)
plt.show()


In [ ]:
# Validación de una imagen individual (adaptación del Colab Test original)
# Cambia la ruta si deseas usar tu propia imagen; si no existe, se usa una imagen de prueba.
ruta_imagen = RAIZ / "imagen_prueba.jpg"
if ruta_imagen.exists():
    imagen = tf.keras.utils.load_img(ruta_imagen, target_size=TAMANIO_IMAGEN)
    imagen = tf.keras.utils.img_to_array(imagen) / 255.0
    imagen_mostrar = imagen
else:
    imagen_tensor, etiqueta_real = next(iter(prueba_sin_batch))
    imagen_mostrar = imagen_tensor.numpy()
    imagen = tf.image.resize(tf.image.convert_image_dtype(imagen_tensor, tf.float32), TAMANIO_IMAGEN).numpy()
imagen_lote = np.expand_dims(imagen, axis=0)
probs = modelo.predict(imagen_lote, verbose=0)[0]
predicha = int(np.argmax(probs))
plt.imshow(np.clip(imagen_mostrar, 0, 255).astype("uint8"))
plt.title(f"Modelo B: {CLASES[predicha]} ({probs[predicha]:.2%})")
plt.axis("off"); plt.show()
print("Clase predicha:", CLASES[predicha])
print(pd.Series(probs, index=CLASES, name="probabilidad").sort_values(ascending=False))

# Si ya existen resultados de los tres modelos, crear comparación y ranking.
rutas = [RAIZ / "results" / f"modelo_{m}_metricas.json" for m in "ABC"]
if all(r.exists() for r in rutas):
    comparacion = pd.DataFrame([json.loads(r.read_text()) for r in rutas])
    for i, fila in comparacion.iterrows():
        meta = json.loads((RAIZ / "results" / f"modelo_{fila['modelo']}_metadata.json").read_text())
        comparacion.loc[i, "parametros_entrenables"] = meta["parametros_entrenables"]
    comparacion.to_csv(RAIZ / "results" / "comparacion_modelos.csv", index=False)
    orden = comparacion.sort_values(
        ["f1_macro", "accuracy", "brecha_f1_validacion_prueba", "parametros_entrenables"],
        ascending=[False, False, True, True],
    )
    ganador = orden.iloc[0]
    resumen = (
        f"El Modelo {ganador['modelo']} obtuvo el mayor F1 macro "
        f"({ganador['f1_macro']:.4f}) y Accuracy {ganador['accuracy']:.4f}."
    )
    (RAIZ / "results" / "ganador.txt").write_text(resumen)
    comparacion.set_index("modelo")[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot(
        kind="bar", figsize=(10, 5), ylim=(0, 1)
    )
    plt.title("Comparación de los tres modelos"); plt.ylabel("Valor")
    plt.xticks(rotation=0); plt.grid(axis="y", alpha=.25); plt.tight_layout()
    plt.savefig(RAIZ / "figures" / "comparacion_modelos.png", dpi=190)
    plt.show()
    print(resumen)
